# Data Diagnosis Toolkit: Colab Runner (Free Tier, GPU ~50%, Drive Persistence)

This notebook runs the complete **Data Diagnosis Toolkit** pipeline and test suite on Google Colab free tier with:
- Google Drive persistence and automatic 25-row checkpointing / resumability
- 50% GPU memory cap (`torch.cuda.set_per_process_memory_fraction(0.5)`)
- Complete test execution (pytest, determinism, privacy canary, harm tests)
- Diagnostic scan, medicine harm testing, and provenance audit generation for `ind-mr-fw2`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/data-diagnosis/ind-mr-fw2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Google Drive mounted successfully at {DRIVE_DIR}")

## 2. Clone Repository and Install Dependencies via UV

In [ ]:
# Clone repo or use uploaded workspace
%cd /content
!rm -rf era-v5-lab
!git clone https://github.com/thinkTinkerGreen/era-v5-lab.git
%cd /content/era-v5-lab

# Install uv for high-speed deterministic dependency management
!pip install -q uv
!uv sync --all-extras
!uv pip install -e .
print("Environment initialized.")

## 3. Run Complete Test Suite on Colab (Per User Requirements)
All unit tests, determinism checks, property tests, and privacy canary tests run on the Colab session.

In [ ]:
# Run full test suite with coverage gate >= 90%
!uv run pytest -v --cov=datadx --cov-fail-under=90 tests/

In [ ]:
# Run determinism contract test (bit-identical outputs)
!uv run pytest -v tests/test_determinism.py

# Run privacy canary test
!uv run pytest -v tests/test_privacy_canary.py

## 4. Execute Full Pipeline on the Sample Pack (`ind-mr-fw2`)
- 50% GPU memory allocation
- Scan for disease candidates
- Confirm & promote clean candidates
- Propose medicines and test harm
- Generate provenance audit Parquet (Rule R9: private to Drive)

In [ ]:
# Run diagnose pipeline
!uv run diagnose run-all ind-mr-fw2.parquet \
    --pack-id ind-mr-fw2 \
    --reads mr \
    --reader thinktinkergreen

# Copy generated artifacts atomically to Google Drive
!cp -r runs/ind-mr-fw2/* /content/drive/MyDrive/data-diagnosis/ind-mr-fw2/
print("Artifacts synced to Google Drive:")
!ls -la /content/drive/MyDrive/data-diagnosis/ind-mr-fw2/

## 5. Verify Submission Artifacts and Agreement

In [ ]:
!ls -la diagnosis/submissions/ind-mr-fw2/thinktinkergreen/
!head -n 20 diagnosis/submissions/ind-mr-fw2/thinktinkergreen/labels.csv